# 📈 Polynomial Regression — Solutions Notebook

**Complete, verified solutions.** Try the practice notebook first!

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete! ✅')

## 🔧 Section 3: Implementation from Scratch

### 3.1 Generate Data

In [ ]:
m = 100
X = np.sort(np.random.uniform(-3, 3, m)).reshape(-1, 1)
y = 1 + 2 * X + 3 * X**2 + np.random.randn(m, 1) * 2

plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.7, edgecolors='k', linewidth=0.5)
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Nonlinear Data (Quadratic)', fontsize=14)
plt.show()

### 3.2 Create Polynomial Features

In [ ]:
# ✅ SOLUTION: Polynomial features from scratch
def create_polynomial_features(X, degree):
    """
    Create polynomial features: [1, x, x², ..., x^d]
    """
    m = X.shape[0]
    X_poly = np.ones((m, 1))  # Start with bias column
    
    for d in range(1, degree + 1):
        X_poly = np.column_stack([X_poly, X ** d])
    
    return X_poly


# Test
X_poly_test = create_polynomial_features(X, degree=3)
assert X_poly_test.shape == (100, 4)
print(f'Degree 3 features shape: {X_poly_test.shape} ✅')
print(f'First row: {X_poly_test[0]}')
print(f'Columns: [1, x, x², x³] = [1, {X[0,0]:.4f}, {X[0,0]**2:.4f}, {X[0,0]**3:.4f}]')

### 3.3 Fit Polynomial Regression

In [ ]:
# ✅ SOLUTION: Fit polynomial using normal equation
def fit_polynomial(X, y, degree):
    """
    Fit polynomial regression using normal equation.
    """
    X_poly = create_polynomial_features(X, degree)
    theta = np.linalg.inv(X_poly.T @ X_poly) @ X_poly.T @ y
    return theta


theta_d2 = fit_polynomial(X, y, degree=2)
print(f'Degree 2 coefficients:')
print(f'  θ₀ = {theta_d2[0, 0]:.4f}  (expected ≈ 1.0)')
print(f'  θ₁ = {theta_d2[1, 0]:.4f}  (expected ≈ 2.0)')
print(f'  θ₂ = {theta_d2[2, 0]:.4f}  (expected ≈ 3.0)')

assert abs(theta_d2[2, 0] - 3.0) < 1.0, "θ₂ should be close to 3.0"
print('\n✅ Polynomial fit looks correct!')

### 3.4 Visualize Underfitting vs Overfitting

In [ ]:
# ✅ SOLUTION: Underfitting vs Overfitting visualization
degrees = [1, 2, 5, 15]
titles = ['Degree 1 (UNDERFITTING)', 'Degree 2 (GOOD FIT)', 
          'Degree 5 (Slight Overfit)', 'Degree 15 (OVERFITTING)']
colors = ['#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

X_smooth = np.linspace(-3, 3, 300).reshape(-1, 1)

for idx, (degree, title, color) in enumerate(zip(degrees, titles, colors)):
    ax = axes[idx // 2, idx % 2]
    
    # Fit polynomial
    theta = fit_polynomial(X, y, degree)
    
    # Predict on smooth X
    X_smooth_poly = create_polynomial_features(X_smooth, degree)
    y_smooth = X_smooth_poly @ theta
    
    # Compute training R²
    X_poly_train = create_polynomial_features(X, degree)
    y_pred_train = X_poly_train @ theta
    ss_res = np.sum((y - y_pred_train) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    
    # Plot
    ax.scatter(X, y, alpha=0.5, edgecolors='k', linewidth=0.3, s=30)
    ax.plot(X_smooth, y_smooth, color=color, linewidth=2.5)
    ax.set_title(f'{title}\nR² = {r2:.4f}', fontsize=12)
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.set_ylim(-10, 40)

plt.suptitle('Underfitting → Good Fit → Overfitting', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print('Key observations:')
print('• Degree 1: Too simple, misses the curve (HIGH BIAS)')
print('• Degree 2: Captures the true pattern perfectly')
print('• Degree 5: Fits well but starts to wiggle')
print('• Degree 15: Memorizes noise, wild oscillations (HIGH VARIANCE)')

---
## 📦 Section 4: Using scikit-learn

In [ ]:
# ✅ SOLUTION: sklearn Pipeline
X_train, X_test, y_train, y_test = train_test_split(
    X, y.ravel(), test_size=0.2, random_state=42
)

pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', LinearRegression())
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'Pipeline (degree=2) Results:')
print(f'  RMSE: {rmse:.4f}')
print(f'  R²:   {r2:.4f}')

---
## 🧪 Section 5: Experiments

### 5.1 Cross-Validation for Degree Selection

In [ ]:
# ✅ SOLUTION: CV for degree selection
degrees_range = range(1, 11)
cv_means = []
cv_stds = []

for d in degrees_range:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])
    scores = cross_val_score(pipe, X, y.ravel(), cv=5, scoring='neg_mean_squared_error')
    cv_means.append(-scores.mean())  # Negate because sklearn returns negative MSE
    cv_stds.append(scores.std())

cv_means = np.array(cv_means)
cv_stds = np.array(cv_stds)

plt.figure(figsize=(10, 6))
plt.errorbar(list(degrees_range), cv_means, yerr=cv_stds, 
             fmt='o-', linewidth=2, capsize=5, capthick=2, markersize=8)
plt.xlabel('Polynomial Degree', fontsize=12)
plt.ylabel('Mean CV MSE', fontsize=12)
plt.title('Cross-Validation Score vs Polynomial Degree', fontsize=14)

best_degree = np.argmin(cv_means) + 1
plt.axvline(x=best_degree, color='r', linestyle='--', alpha=0.7, label=f'Best: degree={best_degree}')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

print(f'\n🏆 Best degree: {best_degree} (CV MSE = {cv_means[best_degree-1]:.4f})')
print(f'\nAll CV MSE values:')
for d, mse_val in zip(degrees_range, cv_means):
    marker = ' ← best' if d == best_degree else ''
    print(f'  Degree {d:2d}: MSE = {mse_val:.4f}{marker}')

### 5.2 Learning Curves

In [ ]:
# ✅ SOLUTION: Learning curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_degrees = [1, 2, 15]
plot_titles = ['Degree 1 (Underfitting)', 'Degree 2 (Good)', 'Degree 15 (Overfitting)']

for ax, d, title in zip(axes, plot_degrees, plot_titles):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])
    
    train_sizes, train_scores, val_scores = learning_curve(
        pipe, X, y.ravel(), 
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring='neg_mean_squared_error'
    )
    
    train_mse = -train_scores.mean(axis=1)
    val_mse = -val_scores.mean(axis=1)
    
    ax.plot(train_sizes, train_mse, 'b-o', label='Training', markersize=4)
    ax.plot(train_sizes, val_mse, 'r-o', label='Validation', markersize=4)
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('MSE')
    ax.set_title(title)
    ax.legend()
    ax.set_ylim(bottom=0)

plt.suptitle('Learning Curves', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('How to read learning curves:')
print('• Underfitting: Both curves plateau at HIGH error → model too simple')
print('• Good fit: Both curves converge to LOW error')
print('• Overfitting: Big GAP between train (low) and val (high) error')

### 5.3 Extrapolation Danger

In [ ]:
# ✅ SOLUTION: Extrapolation danger
theta_d5 = fit_polynomial(X, y, degree=5)

X_wide = np.linspace(-5, 5, 300).reshape(-1, 1)
X_wide_poly = create_polynomial_features(X_wide, degree=5)
y_wide = X_wide_poly @ theta_d5

plt.figure(figsize=(12, 6))
plt.scatter(X, y, alpha=0.6, edgecolors='k', linewidth=0.3, label='Training data', zorder=5)
plt.plot(X_wide, y_wide, 'r-', linewidth=2, label='Degree 5 fit')
plt.axvline(x=-3, color='gray', linestyle='--', alpha=0.7, label='Training range')
plt.axvline(x=3, color='gray', linestyle='--', alpha=0.7)

# Shade extrapolation regions
plt.axvspan(-5, -3, alpha=0.1, color='red', label='Extrapolation zone')
plt.axvspan(3, 5, alpha=0.1, color='red')

plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('⚠️ Polynomial Extrapolation is Dangerous!', fontsize=14)
plt.legend(fontsize=11)
plt.show()

print('Key lesson: Polynomials can behave wildly outside the training range!')
print('Never trust polynomial predictions far from your data.')

---
## 🏆 Section 7: Challenge Solution

In [ ]:
# ✅ SOLUTION: Mystery data challenge
np.random.seed(99)
m_mystery = 80
X_mystery = np.sort(np.random.uniform(-2, 2, m_mystery)).reshape(-1, 1)
y_mystery = (0.5 - X_mystery + 2 * X_mystery**2 - 0.5 * X_mystery**3 
             + np.random.randn(m_mystery, 1) * 0.5)

# Step 1: Cross-validation
best_cv_mse = np.inf
best_d = 1
print('CV Results:')

for d in range(1, 11):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])
    scores = cross_val_score(pipe, X_mystery, y_mystery.ravel(), cv=5, scoring='neg_mean_squared_error')
    mse = -scores.mean()
    marker = ''
    if mse < best_cv_mse:
        best_cv_mse = mse
        best_d = d
        marker = ' ← best so far'
    print(f'  Degree {d:2d}: MSE = {mse:.4f}{marker}')

print(f'\n🏆 Best degree: {best_d}')

# Step 2: Fit best model
theta_mystery = fit_polynomial(X_mystery, y_mystery, best_d)
print(f'\nCoefficients (degree {best_d}):')
for i, t in enumerate(theta_mystery.ravel()):
    print(f'  θ{i} = {t:.4f}')

print(f'\nTrue generating function: y = 0.5 - x + 2x² - 0.5x³')

In [ ]:
# Step 3: Plot the fit
X_smooth = np.linspace(-2, 2, 200).reshape(-1, 1)
X_smooth_poly = create_polynomial_features(X_smooth, best_d)
y_smooth = X_smooth_poly @ theta_mystery

plt.figure(figsize=(10, 6))
plt.scatter(X_mystery, y_mystery, alpha=0.6, edgecolors='k', linewidth=0.3, label='Data')
plt.plot(X_smooth, y_smooth, 'r-', linewidth=2.5, label=f'Fit (degree {best_d})')

# Also plot true function
y_true = 0.5 - X_smooth + 2 * X_smooth**2 - 0.5 * X_smooth**3
plt.plot(X_smooth, y_true, 'g--', linewidth=2, alpha=0.7, label='True function')

plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Mystery Data: Recovered vs True Function', fontsize=14)
plt.legend(fontsize=11)
plt.show()

print('✅ Challenge complete!')

---
## ✅ Summary

- Polynomial regression creates nonlinear features but remains linear in parameters
- Cross-validation is the gold standard for degree selection
- Learning curves diagnose underfitting vs overfitting
- Never extrapolate with polynomials

**Next**: [Ridge/Lasso Regression](../03-ridge-lasso-regression/) →